# Barlow Twins CIFAR-10 Baseline

Barlow Twins training on the shared N-pool, followed by effective-rank, KNN, and top singular-value diagnostics.


In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ModuleNotFoundError:
    print('google.colab is only available inside Google Colab; skipping Drive mount.')


In [ ]:
import getpass
import os
import subprocess
from pathlib import Path
from urllib.parse import quote

PUBLIC_REPO_URL = 'https://github.com/moucheng2017/instance_self_sup.git'
BRANCH = '2-vicreg-barlow-twins-baselines'  # Use 'main' after this baseline suite is merged.
REPO_DIR = Path('/content/instance_self_sup')
GITHUB_TOKEN = ''  # Set to a token string here if the repo is private.


def run(cmd, cwd=None):
    print('+', ' '.join(cmd))
    subprocess.run(cmd, cwd=cwd, check=True)


try:
    from google.colab import userdata
    GITHUB_TOKEN = GITHUB_TOKEN or userdata.get('GITHUB_TOKEN')
except Exception:
    pass

repo_url = PUBLIC_REPO_URL
if GITHUB_TOKEN:
    repo_url = PUBLIC_REPO_URL.replace('https://', f'https://oauth2:{quote(GITHUB_TOKEN, safe="")}@')


def sync_repo():
    if GITHUB_TOKEN:
        run(['git', 'remote', 'set-url', 'origin', repo_url], cwd=REPO_DIR)
    else:
        print('No GitHub token found; keeping the existing origin URL for fetch/pull.')
    run(['git', 'fetch', 'origin', BRANCH], cwd=REPO_DIR)
    run(['git', 'checkout', BRANCH], cwd=REPO_DIR)
    run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO_DIR)

os.chdir('/content')

if not REPO_DIR.exists():
    run(['git', 'clone', '--branch', BRANCH, repo_url, str(REPO_DIR)])
else:
    sync_repo()

os.chdir(REPO_DIR)
run(['python', '-m', 'pip', 'install', '-r', 'requirements_colab.txt'])
print('Repository is ready at', REPO_DIR)


In [ ]:
# CIFAR-10 is downloaded automatically by torchvision the first time this cell runs.
import os
import torchvision

DATA_DIR = '/content/drive/MyDrive/SSL_exps/data'
os.makedirs(DATA_DIR, exist_ok=True)

print('Checking / downloading CIFAR-10 ...')
torchvision.datasets.CIFAR10(DATA_DIR, train=True, download=True)
torchvision.datasets.CIFAR10(DATA_DIR, train=False, download=True)
print('CIFAR-10 is ready.')


## Sweep Parameters

| Parameter | Description |
|---|---|
| `N_SWEEP`, `subset_n`, `subset_seed` | Shared low-data pool for training and diagnostics |
| `DEFAULT_BATCH_SIZE`, `NUM_EPOCHS`, `STOP_AT_EPOCH` | Per-N training controls; batch size is clamped to `N` |
| `INIT_CHECKPOINT`, `INIT_LOAD_BACKBONE`, `INIT_LOAD_PROJECTOR`, `INIT_LOAD_PREDICTOR` | Optional checkpoint initialization controls |
| `knn_monitor` / `monitor_accuracy` | In-training monitor is disabled by default; Section 2.1 KNN is computed in the diagnostics cell |
| `lambd`, `projector_dim` | Barlow Twins off-diagonal coefficient and CIFAR-scale projector width |


In [ ]:
from colab_utils import train_from_colab

CONFIG_FILE = 'configs/baselines/barlow_twins_cifar10.yaml'  # config_file='configs/baselines/barlow_twins_cifar10.yaml'
PROJECT_NAME = 'SSL_exps'
SUBSET_SEED = 42
N_SWEEP = [200, 500, 1000, 5000, 'full']
DEFAULT_BATCH_SIZE = 256
NUM_EPOCHS = 100
STOP_AT_EPOCH = 100

INIT_CHECKPOINT = None
INIT_LOAD_BACKBONE = True
INIT_LOAD_PROJECTOR = False
INIT_LOAD_PREDICTOR = False

LAMBD = 0.0051
PROJECTOR_DIM = 2048


def batch_size_for_n(n):
    if n == 'full' or n is None:
        return DEFAULT_BATCH_SIZE
    return min(DEFAULT_BATCH_SIZE, int(n))


results = []
for n in N_SWEEP:
    subset_n = None if n == 'full' else int(n)
    batch_size = batch_size_for_n(n)
    overrides = {
        'name': f'{CONFIG_FILE.split("/")[-1].replace(".yaml", "")}-N{n}',
        'train': {
            'subset_n': subset_n,
            'subset_seed': SUBSET_SEED,
            'batch_size': batch_size,
            'num_epochs': NUM_EPOCHS,
            'stop_at_epoch': STOP_AT_EPOCH,
            'knn_monitor': False,
        },
        'model': {
            'init_checkpoint': INIT_CHECKPOINT,
            'init_load_backbone': INIT_LOAD_BACKBONE,
            'init_load_projector': INIT_LOAD_PROJECTOR,
            'init_load_predictor': INIT_LOAD_PREDICTOR,
        'lambd': LAMBD,
        'projector_dim': PROJECTOR_DIM,
        },
    }
    run_result = train_from_colab(
        config_file=CONFIG_FILE,
        project_name=PROJECT_NAME,
        overrides=overrides,
        device='cuda',
        download=True,
        hide_progress=False,
    )
    results.append({
        'N': n,
        'subset_n': subset_n,
        'subset_seed': SUBSET_SEED,
        'batch_size': batch_size,
        'model_path': run_result['model_path'],
        'log_dir': run_result['log_dir'],
        'selected_subset_indices_path': run_result.get('selected_subset_indices_path'),
        'monitor_accuracy': run_result.get('accuracy'),
    })

results


In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd
import torch

from analysis.spectral import build_diagnostics_loader, extract_features, knn_eval, spectral_diagnostics
from arguments import build_args
from datasets.subset import load_subset_indices, select_subset_indices
from linear_eval import load_backbone_weights
from models import get_backbone


def selected_indices_for_result(args, item):
    selected_path = item.get('selected_subset_indices_path')
    if selected_path:
        return load_subset_indices(selected_path)
    if item['subset_n'] is None:
        return None
    probe_loader = build_diagnostics_loader(args, indices=None, batch_size=item['batch_size'])
    return select_subset_indices(len(probe_loader.dataset), item['subset_n'], item['subset_seed'])


def run_section_21_diagnostics(config_file, sweep_results):
    rows = []
    singular_values_by_n = {}
    for item in sweep_results:
        overrides = {
            'train': {
                'subset_n': item['subset_n'],
                'subset_seed': item['subset_seed'],
                'batch_size': item['batch_size'],
            }
        }
        args = build_args(
            config_file=config_file,
            overrides=overrides,
            data_dir=DATA_DIR,
            log_dir='/tmp/diagnostics_logs',
            ckpt_dir='/tmp/diagnostics_ckpts',
            device='cuda' if torch.cuda.is_available() else 'cpu',
            download=False,
            create_dirs=False,
        )
        selected_indices = selected_indices_for_result(args, item)
        loader = build_diagnostics_loader(args, indices=selected_indices, batch_size=item['batch_size'])
        backbone = get_backbone(args.model.backbone).to(args.device)
        load_backbone_weights(backbone, item['model_path'])
        n_samples = len(loader.dataset) if item['subset_n'] is None else item['subset_n']
        features, labels = extract_features(backbone, loader, args.device, n_samples=n_samples)
        singular_values, erank, explained_variance = spectral_diagnostics(features)
        n_train = int(0.8 * len(features))
        section21_knn = knn_eval(features, labels, k=min(20, n_train), n_train=n_train)
        rows.append({
            'N': item['N'],
            'effective_rank': erank,
            'knn_accuracy': section21_knn,
            'monitor_accuracy': item.get('monitor_accuracy'),
            'checkpoint': item['model_path'],
        })
        singular_values_by_n[item['N']] = singular_values[:20]
    return pd.DataFrame(rows), singular_values_by_n


diagnostics_table, singular_values_by_n = run_section_21_diagnostics(CONFIG_FILE, results)
display(diagnostics_table)

plt.figure(figsize=(8, 5))
for n, values in singular_values_by_n.items():
    plt.plot(range(1, len(values) + 1), values, marker='o', label=f'N={n}')
plt.xlabel('Singular value rank')
plt.ylabel('Singular value')
plt.title('Top-20 penultimate-feature singular values')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()
